# 03 — ML Question

Owner: Khoi (Q2). Data loaded from `data/processed/` (see ADR-003).

Authority order: `ADR > SPECIFICATION.md > notebook`.

## Decisions taken (2026-11-26)

| # | Decision | Chosen | Why |
| --- | --- | --- | --- |
| D1 | Models | **Ridge + HistGradientBoosting** | One interpretable linear reference, one nonlinear learner. Divergent failure modes make the error analysis readable. |
| D2 | Grids | Ridge `alpha` {0.01…100}; HGB `max_depth` {2,3,None} x `min_samples_leaf` {1,5,10} | Wide enough to show the search *prefers* constrained capacity at n=54, without letting best-params drift across folds. |
| D3 | CV | `TimeSeriesSplit(n_splits=4)` | First training fold ≈13 rows (safe for the models); deeper splits leave early folds too thin. Ordering is asserted per fold rather than assumed. |
| D4 | Country | **Pooled primary**, country dummies as D6 robustness | Answers "can one rule predict across six countries?" — the harder and more interesting claim. `sales_lag1` already encodes each country's level, so dummies mostly re-state it; the RMSE gap between the two is itself a finding. |
| D5 | Error cuts | by country, by target value range, by policy regime (2021-22 vs 2023-24) | Value range shows where growth was fastest; merged regimes keep cells large enough at n=24 to test the SPECIFICATION §2 regime-change warning. |

---

## Q2 — EV adoption diffusion (ML)

### Task definition

*Your words here — the brief requires a named user and the decision this informs.*

The pre-registered task (SPECIFICATION §3 Q2): predict `EV sales share` for year
**t+1** from information available at year **t**, `mode = Cars`,
`powertrain = EV`, `category = Historical`.

In [ ]:
# Setup — imports, paths, constants
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path("..").resolve() if not Path("data").exists() else Path(".").resolve()
PROCESSED = ROOT / "data" / "processed"

SEED = 2
print("ROOT:", ROOT)

### Load

Row counts are asserted, not assumed: `prepare_data.py` pins 61 / 496.

In [ ]:
ev = pd.read_csv(PROCESSED / "ev_long.csv")

print("ev_long:", ev.shape)
assert len(ev) == 496, f"expected 496 rows, got {len(ev)}"
assert set(ev["parameter"].unique()) == {
    "EV sales share", "EV stock share", "EV charging points"
}, "unexpected parameter set"

# prepare_data.py keeps only category = Historical, so there is no
# Projection-STEPS row available to leak IEA's own model output into the
# target (SPECIFICATION §3 leakage guard).
print("years:", ev["year"].min(), "-", ev["year"].max())
print("parameters:", sorted(ev["parameter"].unique()))

### Panel construction

Two exclusions, both pre-registered and both reported with their row-count
consequence — never applied silently:

- **Singapore** — absent from the IEA file (ADR-002 C.6).
- **Vietnam** — `cars` exists only for 2018 and 2021-2024, so with a 2-year lag
  it contributes **0 training rows** (ADR-008 D.5). The two-wheeler evidence
  lives in Q4.

Remaining sample: 6 countries.

In [ ]:
COUNTRIES = ["China", "Germany", "Japan", "New Zealand", "Thailand", "United States"]
EXCLUDED = {
    "Singapore": "absent from the IEA file (ADR-002 C.6)",
    "Vietnam": "cars only 2018, 2021-2024 -> 0 training rows after a 2-year lag (ADR-008 D.5)",
}

def series(param, mode):
    '''country x year wide table for one (parameter, mode) pair.'''
    sub = ev[(ev["parameter"] == param) & (ev["mode"] == mode)]
    return sub.pivot_table(index="country", columns="year", values="value").loc[COUNTRIES]

sales = series("EV sales share", "Cars")
stock = series("EV stock share", "Cars")
charger = series("EV charging points", "EV")

print("countries kept:", len(COUNTRIES), "| excluded:", len(EXCLUDED))
for c, why in EXCLUDED.items():
    print(f"  - {c}: {why}")
print()
print("non-null years per country:")
for name, w in [("sales", sales), ("stock", stock), ("charger", charger)]:
    print(f"  {name:8s}", {c: int(w.loc[c].notna().sum()) for c in COUNTRIES})

### Controls (World Bank)

`log GDP per capita` and `population` are needed at the **origin** year, so this
fetches 2010-2024 — wider than the 2018/2024 pair Q1 uses.

New supporting source, so ADR-002 D.7 applies: licence, access route and
download date are recorded below. **Caching is not pinning** — see #41 item 10.

In [ ]:
WB_CACHE = PROCESSED / "world_bank_q2.json"
ACCESS_DATE = "2026-11-26"      # recorded; a fresh clone re-fetches live data
WB_LICENCE = "CC BY 4.0"        # https://data.worldbank.org/

ISO3 = {"China": "CHN", "Germany": "DEU", "Japan": "JPN",
        "New Zealand": "NZL", "Thailand": "THA", "United States": "USA"}

def fetch_world_bank():
    codes = ";".join(ISO3.values())
    inv = {v: k for k, v in ISO3.items()}
    out = {}
    for key, ind in [("gdp", "NY.GDP.PCAP.CD"), ("pop", "SP.POP.TOTL")]:
        url = (f"https://api.worldbank.org/v2/country/{codes}/indicator/{ind}"
               f"?format=json&date=2010:2024&per_page=2000")
        payload = requests.get(url, timeout=30).json()
        rows = {}
        for rec in payload[1]:
            name = inv.get(rec["countryiso3code"])
            if name and rec["value"] is not None:
                rows[f"{name},{int(rec['date'])}"] = float(rec["value"])
        out[key] = rows
    return out

if WB_CACHE.exists():
    wb = json.loads(WB_CACHE.read_text())
    print("loaded cached World Bank data")
else:
    wb = fetch_world_bank()
    WB_CACHE.write_text(json.dumps(wb, indent=1))
    print("fetched and cached World Bank data")

print(f"licence: {WB_LICENCE} | accessed: {ACCESS_DATE}")
print("gdp cells:", len(wb["gdp"]), "| pop cells:", len(wb["pop"]))

### Features and the origin/target mapping

`origin` = the year the prediction is made from. Target = `sales[origin + 1]`.

| Feature | Lag vs target |
| --- | --- |
| `sales[origin]` | 1 year |
| `sales[origin - 1]` | 2 years |
| `stock[origin]` | 1 year |
| `charger_per_1000[origin]` | 1 year |
| `log_gdp[origin]` | 1 year |

Country identity is **not** a feature in the primary model (D4). The 2-year lag
is why origins start at 2011.

In [ ]:
origins = range(2011, 2024)         # 2023 + 1 = 2024, the last target year available

records = []
for country in COUNTRIES:
    for t in origins:
        pop = wb["pop"].get(f"{country},{t}")
        gdp = wb["gdp"].get(f"{country},{t}")
        records.append({
            "country": country,
            "origin": t,
            "target_year": t + 1,
            "sales_lag1": sales.loc[country, t],
            "sales_lag2": sales.loc[country, t - 1],
            "stock_lag1": stock.loc[country, t],
            "charger_per_1000": (charger.loc[country, t] * 1000.0 / pop) if pop else np.nan,
            "log_gdp": np.log(gdp) if gdp else np.nan,
            "y_target": sales.loc[country, t + 1],
        })

panel = pd.DataFrame(records)
print("candidate rows:", len(panel))

before = len(panel)
panel = panel.dropna(subset=["y_target", "sales_lag1", "sales_lag2", "stock_lag1", "log_gdp"])
print(f"dropped {before - len(panel)} rows with a missing target/lag/control -> {len(panel)}")

# Chargers start in different years per country. NaN is expected and is imputed
# *inside* the Pipeline, fit on train only (SPECIFICATION §3 Q2).
print("charger_per_1000 NaN rows:", int(panel["charger_per_1000"].isna().sum()))

# Sort by ORIGIN first. TimeSeriesSplit cuts the row order, so grouping by
# country would make folds run through one country's years instead of through
# time -- each fold would train on years after the years it tests.
panel = panel.sort_values(["origin", "country"]).reset_index(drop=True)

# D4 robustness: country dummies, built on the PANEL (before the split). Doing
# this per-split would produce mismatched columns if a country were absent from
# one side. Country identity is known at prediction time, so this is not leakage.
dummies = pd.get_dummies(panel["country"], prefix="c", drop_first=True)
panel = pd.concat([panel, dummies], axis=1)

print("first rows (origin, country):")
print(panel[["origin", "country"]].head(6).to_string(index=False))
print("dummy columns:", list(dummies.columns))

### Train/test split (train <= 2020, test 2021-2024)

Time-based on the **origin**, so every test target year is strictly later than
every training target year. Expected 54 train (9 origins x 6 countries) and
24 test (4 origins x 6 countries), per SPECIFICATION §2.

In [ ]:
TRAIN_ORIGIN_MAX = 2019          # targets <= 2020

train = panel[panel["origin"] <= TRAIN_ORIGIN_MAX].copy().reset_index(drop=True)
test = panel[panel["origin"] > TRAIN_ORIGIN_MAX].copy().reset_index(drop=True)

print(f"train: {len(train)} rows | origins {train['origin'].min()}-{train['origin'].max()}"
      f" | targets {train['target_year'].min()}-{train['target_year'].max()}")
print(f"test : {len(test)} rows | origins {test['origin'].min()}-{test['origin'].max()}"
      f" | targets {test['target_year'].min()}-{test['target_year'].max()}")

assert len(train) == 54, f"expected 54 train rows, got {len(train)}"
assert len(test) == 24, f"expected 24 test rows, got {len(test)}"
assert train["target_year"].max() < test["target_year"].min(), "train/test target years overlap"
print()
print("train rows per country:", train.groupby("country").size().to_dict())
print("test  rows per country:", test.groupby("country").size().to_dict())

### Leakage audit

The brief requires the excluded columns to be **listed**, not merely avoided.

In [ ]:
FEATURES = ["sales_lag1", "sales_lag2", "stock_lag1", "charger_per_1000", "log_gdp"]
COUNTRY_DUMMIES = [c for c in panel.columns if c.startswith("c_")]
FEATURES_DUMMIED = FEATURES + COUNTRY_DUMMIES

EXCLUDED_COLUMNS = {
    "sales[target_year]": "the target itself",
    "stock[target_year]": "contemporaneous with the target",
    "charger[target_year]": "contemporaneous with the target",
    "gdp[target_year]": "contemporaneous with the target",
    "category=Projection-STEPS": "IEA model output -> leaks IEA's own projection (SPECIFICATION §3)",
    "sales share, other modes": "only Cars is the pinned target definition",
}

for f in FEATURES:
    assert f in train.columns, f"feature {f} missing"
assert not any(f.startswith("y_") for f in FEATURES), "target leaked into FEATURES"
assert "target_year" not in FEATURES and "origin" not in FEATURES

print("primary features:", FEATURES)
print("dummied features:", FEATURES_DUMMIED)
print()
print("excluded and why:")
for col, why in EXCLUDED_COLUMNS.items():
    print(f"  - {col:28s} {why}")

### Baseline — persistence

The trivial baseline the brief requires: predict that next year equals this
year. `persistence = sales[origin]`, i.e. the `sales_lag1` feature.

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def mae(y_true, y_pred):
    return float(mean_absolute_error(y_true, y_pred))

y_train = train["y_target"].values
y_test = test["y_target"].values

BASELINE_RMSE = rmse(y_test, test["sales_lag1"].values)
BASELINE_MAE = mae(y_test, test["sales_lag1"].values)
print(f"persistence  test RMSE = {BASELINE_RMSE:.4f} pp   MAE = {BASELINE_MAE:.4f} pp")

# SPECIFICATION §5 Q2: success = test RMSE at least 1.0 pp below the baseline.
THRESHOLD_PP = 1.0
print(f"bar to beat: {BASELINE_RMSE - THRESHOLD_PP:.4f} pp")

### Models and grid (D1, D2)

Both pipelines keep preprocessing **inside** the `Pipeline`, so the median
imputer is fit on training folds only. Ridge additionally scales (dummy columns
in the robustness run need it); HGB does not require scaling.

In [ ]:
CANDIDATES = {
    "ridge": {
        "estimator": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
            ("model", Ridge()),
        ]),
        "grid": {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    },
    "hist_gb": {
        "estimator": Pipeline([
            ("impute", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(random_state=SEED)),
        ]),
        "grid": {"model__max_depth": [2, 3, None],
                 "model__min_samples_leaf": [1, 5, 10]},
    },
}

N_SPLITS = 4          # D3
SCORING = "neg_root_mean_squared_error"
print("candidates:", list(CANDIDATES), "| folds:", N_SPLITS)

### Evaluation — tune on train only, test once

`TimeSeriesSplit` alone is **not** correct for this panel: it splits on row
positions, and each origin owns six rows, so an index-based cut lands inside an
origin year and that year appears in both train and validation. `OriginFolds`
below splits on the unique origin years instead, then the per-fold assert checks
the property rather than assuming it.

Note the first fold trains on a single origin (6 rows). That is inherent to 9
origins and 4 folds; it is reported here so the thin fold is visible rather than
silent.

In [ ]:
TEST_SIZE = 2      # origins held out per validation fold

class OriginFolds:
    """Time-ordered folds that never split INSIDE an origin year.

    TimeSeriesSplit works on row positions. Each origin contributes six rows
    (one per country), so an index-based split lands in the middle of an origin
    year and that boundary year ends up in both train and validation. This
    splits on the unique origin years instead, which is what "time-based" has
    to mean for a panel.
    """
    def __init__(self, frame, n_splits, test_size):
        self.origins = frame["origin"].to_numpy()
        self.years = np.array(sorted(set(self.origins.tolist())))
        self.n_splits = n_splits
        self.test_size = test_size

    def split(self, X, y=None, groups=None):
        assert len(X) == len(self.origins), "row order mismatch between X and folds"
        tscv = TimeSeriesSplit(n_splits=self.n_splits, test_size=self.test_size)
        for tr_y, va_y in tscv.split(self.years):
            yield (np.flatnonzero(np.isin(self.origins, self.years[tr_y])),
                   np.flatnonzero(np.isin(self.origins, self.years[va_y])))

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


FOLDS = OriginFolds(train, N_SPLITS, TEST_SIZE)


def check_folds(folds):
    """Every fold must train on origins strictly earlier than it validates."""
    for k, (tr, va) in enumerate(folds.split(train)):
        tr_max = train.iloc[tr]["origin"].max()
        va_min = train.iloc[va]["origin"].min()
        assert tr_max < va_min, (
            f"fold {k}: trains to origin {tr_max} but validates from {va_min} -- "
            "the split fell inside an origin year")
        print(f"  fold {k}: train {len(tr):2d} rows "
              f"(origins {train.iloc[tr]['origin'].min()}-{tr_max})"
              f" | validate {len(va):2d} rows "
              f"(origins {va_min}-{train.iloc[va]['origin'].max()})")
    print("ordering OK: every fold trains strictly before it validates")


check_folds(FOLDS)


def evaluate(label, features, tr, te, y_tr, y_te, store_preds=False):
    """GridSearchCV on the training set only, then a one-shot test evaluation."""
    out = {}
    for name, spec in CANDIDATES.items():
        search = GridSearchCV(spec["estimator"], spec["grid"], cv=FOLDS, scoring=SCORING)
        search.fit(tr[features], y_tr)
        pred = search.predict(te[features])
        out[name] = {"rmse": rmse(y_te, pred), "mae": mae(y_te, pred),
                     "cv_rmse": -search.best_score_, "best_params": search.best_params_}
        if store_preds:
            out[name]["pred"] = pred
        print(f"{label}/{name:8s} cv={out[name]['cv_rmse']:.4f} "
              f"test_rmse={out[name]['rmse']:.4f} test_mae={out[name]['mae']:.4f} "
              f"{search.best_params_}")
    return out


print("--- pooled (primary, D4) ---")
pooled = evaluate("pooled", FEATURES, train, test, y_train, y_test, store_preds=True)

### D4 robustness — country dummies

Same two models, same grids, only the feature list changes: the five dummy
columns are added. Nothing else moves.

The **gap** between the pooled and dummied RMSE is the finding: it says how much
of the predictability lives at the country level versus within a country over
time. A small gap means the dynamics alone carry it; a large one means the model
mostly needed to know which country it was looking at.

In [ ]:
train_d = train.copy()
test_d = test.copy()

print("--- pooled + country dummies (D4 robustness) ---")
dummied = evaluate("dummied", FEATURES_DUMMIED, train_d, test_d, y_train, y_test)

print("\n=== results (test RMSE, pp) ===")
rows = {"persistence": {"rmse": BASELINE_RMSE, "mae": BASELINE_MAE}}
for name in CANDIDATES:
    rows[f"pooled/{name}"] = {"rmse": pooled[name]["rmse"], "mae": pooled[name]["mae"]}
    rows[f"dummied/{name}"] = {"rmse": dummied[name]["rmse"], "mae": dummied[name]["mae"]}
table = pd.DataFrame(rows).T[["rmse", "mae"]].round(4)
print(table.to_string())

print("\nD4 gap (pooled minus dummied test RMSE; positive = dummies helped):")
for name in CANDIDATES:
    gap = pooled[name]["rmse"] - dummied[name]["rmse"]
    print(f"  {name:8s} {gap:+.4f} pp")

print(f"\nQ2 verdict (SPECIFICATION §5): bar = {BASELINE_RMSE - THRESHOLD_PP:.4f} pp")
for name in CANDIDATES:
    ok = pooled[name]["rmse"] <= BASELINE_RMSE - THRESHOLD_PP
    print(f"  pooled/{name:8s} {'MET' if ok else 'NOT MET'} "
          f"({pooled[name]['rmse']:.4f} vs {BASELINE_RMSE - THRESHOLD_PP:.4f})")

### Errors — where does it fail (D5)

Three cuts, as decided: by country, by target value range, and by policy regime.

*Your reading of these tables is the answer to the question.* The code produces
them; the interpretation is yours.

Note on which model's errors these are: the **primary model for error analysis is
chosen by cross-validated RMSE on the training set**, not by test performance —
picking the best test score and then analysing that model's errors would be
selecting on the outcome.

In [ ]:
# Choose the model to analyse by TRAIN-only CV, never by test score.
primary_name = min(pooled, key=lambda k: pooled[k]["cv_rmse"])
print(f"primary model (lowest train CV RMSE): {primary_name}")

test = test.copy()
test["pred"] = pooled[primary_name]["pred"]
test["abs_err"] = (test["y_target"] - test["pred"]).abs()
test["resid"] = test["y_target"] - test["pred"]

print(f"\noverall test RMSE = {pooled[primary_name]['rmse']:.4f} pp "
      f"| persistence = {BASELINE_RMSE:.4f} pp")

print("\n[1] by country:")
by_country = test.groupby("country")["abs_err"].agg(["mean", "max", "count"]).round(3)
by_country["bias"] = test.groupby("country")["resid"].mean().round(3)
print(by_country.sort_values("mean", ascending=False).to_string())

print("\n[2] by target value range (pp):")
bands = pd.cut(test["y_target"], bins=[-1, 5, 15, 30, 100],
               labels=["<5", "5-15", "15-30", ">30"])
print(test.groupby(bands, observed=False)["abs_err"].agg(["mean", "max", "count"]).round(3).to_string())

print("\n[3] by policy regime (D5: merged years to keep cells large):")
regime = np.where(test["target_year"] <= 2022, "2021-22", "2023-24")
test["regime"] = regime
print(test.groupby("regime")["abs_err"].agg(["mean", "max", "count"]).round(3).to_string())

### Interpretation

*Your words here.* Two plots minimum (brief §3.4): at least one should show the
prediction-versus-actual story, one the error structure.

`permutation_importance` below is **indicative only** — with 24 test rows the
repeats are noisy, and collinear features (`sales_lag1` / `sales_lag2`) tend to
split their importance arbitrarily. Say that in the write-up rather than quoting
the ranking as fact.

In [ ]:
spec = CANDIDATES[primary_name]
best = GridSearchCV(spec["estimator"], spec["grid"],
                    cv=FOLDS,
                    scoring=SCORING).fit(train[FEATURES], y_train).best_estimator_

imp = permutation_importance(best, test[FEATURES], y_test, n_repeats=50,
                             random_state=SEED, scoring=SCORING)
imp_df = (pd.DataFrame({"feature": FEATURES,
                        "mean": imp.importances_mean,
                        "std": imp.importances_std})
          .sort_values("mean", ascending=False))
print(f"permutation importance ({primary_name}), n_repeats=50, n=24 — indicative:")
print(imp_df.round(4).to_string(index=False))

# Plot 1: predicted vs actual
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.scatter(test["y_target"], test["pred"], s=45, edgecolor="k", linewidth=0.4, zorder=3)
lim = [0, max(test["y_target"].max(), test["pred"].max()) * 1.05]
ax.plot(lim, lim, "r--", lw=1, label="perfect prediction")
ax.set_xlabel("actual EV sales share, next year (pp)")
ax.set_ylabel("predicted (pp)")
ax.set_title(f"{primary_name}: predicted vs actual (test 2021-2024)")
ax.legend()
ax.grid(alpha=0.3)

# Plot 2: error by country
ax = axes[1]
order = by_country.sort_values("mean", ascending=False).index
ax.bar(range(len(order)), by_country.loc[order, "mean"], color="#1f77b4", edgecolor="k", linewidth=0.4)
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order, rotation=30, ha="right")
ax.set_ylabel("mean absolute error (pp)")
ax.set_title("Where it fails: error by country")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

### Still to write (yours, not the scaffold's)

- The **task definition** paragraph: named user, decision informed.
- The **metric justification**: why RMSE in percentage points, and why
  R-squared alone would mislead when country levels differ by two orders of
  magnitude.
- The **error reading**: what the three cuts say, especially whether the policy
  regime split supports or contradicts SPECIFICATION §2's regime-change warning.
- The **answer to Q2**: is cross-country diffusion predictable, and where does
  the model fail?
- Two captioned plots.